# ReSpace-AI: Interior Design Inference on Kaggle
This notebook provides a Gradio web interface for generating interior design images using SD 1.5 LoRAs. It uses `diffusers` for the pipeline and `gradio` for the UI.

In [20]:
# 1. Install and Import Required Libraries
!pip install -q diffusers transformers accelerate safetensors gradio pillow torch omegaconf

import os
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline
from transformers import CLIPTextModel, CLIPTokenizer
from PIL import Image
import glob

In [30]:
# 2. Environment Configuration and Constants
import glob

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Define UI Constants
ROOM_TYPES = ["Bedroom", "Kitchen", "Lounge", "Dining Room"]
COLOR_THEMES = [
    "Modern White", "Sleek Charcoal", "Earthy Terracotta", 
    "Royal Blue", "Soft Beige", "Emerald Green", "Pastel Pink"
]
COLOR_PALETTES = ["Neutral", "Warm", "Pastel", "Dark Mode"]
LIGHTING_CONDITIONS = ["Natural Daylight", "Warm Indoor Lighting", "LED Office Lighting"]
FURNITURE_TYPES = ["Desks", "Sofas", "Shelves", "Table", "Chair", "Bed", "Lighting Fixtures", "Plants", "Partitions"]
USE_CASES = {
    "Bedroom": ["Kids", "Couple", "Adults", "Guest", "Master Suite", "Minimalist Retreat"],
    "Kitchen": ["Professional Chef", "Small Family", "Open Concept", "Modern Minimalist", "Industrial Loft"],
    "Lounge": ["Home Theater", "Formal Entertaining", "Cozy Family", "Reading Nook", "Social Hub"],
    "Dining Room": ["Formal", "Casual Family", "Bistro Style", "Elegant Party", "Rustic"]
}

# Base Model Paths (3 separate engines)
SD15_BASE  = "runwayml/stable-diffusion-v1-5"         # Vanilla SD 1.5
EPIC_BASE  = "emilianJR/epiCRealism"                  # epiCRealism (SD 1.5 fine-tune, separate engine)
SD35_BASE  = "tensorart/stable-diffusion-3.5-medium-turbo"  # SD 3.5 Medium Turbo

# --- SMART PATH DISCOVERY ---
def find_dataset_root():
    # Search for any path that ends with 'respace-ai-models'
    potential_paths = glob.glob("/kaggle/input/**/respace-ai-models", recursive=True)
    if potential_paths:
        return potential_paths[0]
    return "/kaggle/input/respace-ai-models"  # Fallback

DATASET_ROOT = find_dataset_root()
print(f"✓ Dataset Root Found: {DATASET_ROOT}")

# Each engine has its own LoRA directory
# Each engine has its own LoRA directory
SD15_MODELS_DIR = f"{DATASET_ROOT}/models/sd1.5"         # SD 1.5 LoRAs  (was "sd15")
EPIC_MODELS_DIR = f"{DATASET_ROOT}/models/epicrealism"   # epiCRealism LoRAs (correct)
SD35_LORA_PATH  = f"{DATASET_ROOT}/sd35-medium-turbo/sd35-medium-turbo"  # nested folder fix

print(f"SD 1.5 Models Path     : {SD15_MODELS_DIR}")
print(f"epiCRealism Models Path: {EPIC_MODELS_DIR}")
print(f"SD 3.5 Models Path     : {SD35_LORA_PATH}")

# --- Verify All Three Folders ---
print("\n--- Verifying Model Folders ---")

if os.path.exists(SD15_MODELS_DIR):
    print(f"✓ Found SD 1.5 folder      : {os.listdir(SD15_MODELS_DIR)}")
else:
    print(f"✗ SD 1.5 folder NOT found at {SD15_MODELS_DIR}")

if os.path.exists(EPIC_MODELS_DIR):
    print(f"✓ Found epiCRealism folder : {os.listdir(EPIC_MODELS_DIR)}")
else:
    print(f"✗ epiCRealism folder NOT found at {EPIC_MODELS_DIR}")

if os.path.exists(SD35_LORA_PATH):
    print(f"✓ Found SD 3.5 folder      : {os.listdir(SD35_LORA_PATH)}")
else:
    print(f"✗ SD 3.5 folder NOT found at {SD35_LORA_PATH}")

Using device: cuda
✓ Dataset Root Found: /kaggle/input/datasets/fatima753/respace-ai-models
SD 1.5 Models Path     : /kaggle/input/datasets/fatima753/respace-ai-models/models/sd1.5
epiCRealism Models Path: /kaggle/input/datasets/fatima753/respace-ai-models/models/epicrealism
SD 3.5 Models Path     : /kaggle/input/datasets/fatima753/respace-ai-models/sd35-medium-turbo/sd35-medium-turbo

--- Verifying Model Folders ---
✓ Found SD 1.5 folder      : ['respace_interior_lora_v3_dadapt.safetensors', 'respace_interior_lora_fast.safetensors', 'respace_interior_lora_v2_highdim.safetensors']
✓ Found epiCRealism folder : ['lora_high_dim.safetensors', 'last.safetensors', 'lora_aggressive.safetensors']
✓ Found SD 3.5 folder      : ['base-fine-tune.safetensors', 'increased_rank.safetensors', 'increased_steps.safetensors']


In [31]:
# 3. Model Loading and Multi-Engine Pipeline Setup
import gc
from diffusers import StableDiffusionPipeline, StableDiffusion3Pipeline

# Global variables for the current engine state
current_engine = None  # "SD 1.5", "epiCRealism", or "SD 3.5"
pipe = None

def load_engine(engine_type):
    global pipe, current_engine

    if current_engine == engine_type and pipe is not None:
        return

    print(f"Switching engine to {engine_type}...")
    # Clean up old pipe to save VRAM
    if pipe is not None:
        del pipe
        torch.cuda.empty_cache()
        gc.collect()

    try:
        if engine_type == "SD 1.5":
            pipe = StableDiffusionPipeline.from_pretrained(
                SD15_BASE,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                safety_checker=None
            ).to(device)

        elif engine_type == "epiCRealism":
            pipe = StableDiffusionPipeline.from_pretrained(
                EPIC_BASE,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                safety_checker=None
            ).to(device)

        else:  # SD 3.5
            pipe = StableDiffusion3Pipeline.from_pretrained(
                SD35_BASE,
                torch_dtype=torch.float16,
                device_map="balanced"
            )

        current_engine = engine_type
        print(f"✓ Engine loaded: {engine_type}")

    except Exception as e:
        print(f"✗ Error loading {engine_type}: {e}")


def load_lora_to_pipe(engine_type, lora_name):
    global pipe
    load_engine(engine_type)

    if pipe is None:
        return f"Error: Pipeline failed to load for {engine_type}"

    try:
        if hasattr(pipe, "unload_lora_weights"):
            pipe.unload_lora_weights()

        if engine_type == "SD 1.5":
            lora_path = os.path.join(SD15_MODELS_DIR, lora_name)
            pipe.load_lora_weights(lora_path)
            label = f"SD 1.5 (base: {SD15_BASE})"

        elif engine_type == "epiCRealism":
            lora_path = os.path.join(EPIC_MODELS_DIR, lora_name)
            pipe.load_lora_weights(lora_path)
            label = f"epiCRealism (base: {EPIC_BASE})"

        else:  # SD 3.5
            pipe.load_lora_weights(lora_name)
            label = f"SD 3.5 Turbo (base: {SD35_BASE})"

        return f"✓ {label} + LoRA: {os.path.basename(lora_name)} ready!"

    except Exception as e:
        return f"✗ Error loading LoRA: {e}"


def get_lora_list():
    items = []

    # SD 1.5 LoRAs
    if os.path.exists(SD15_MODELS_DIR):
        print(f"Scanning SD 1.5 : {SD15_MODELS_DIR}")
        for root, _, files in os.walk(SD15_MODELS_DIR):
            for f in files:
                if ".safe" in f.lower() or f.endswith(".bin"):
                    rel_path = os.path.relpath(os.path.join(root, f), SD15_MODELS_DIR)
                    items.append((f"SD 1.5: {rel_path}", "SD 1.5", rel_path))
    else:
        print(f"✗ SD 1.5 folder not found: {SD15_MODELS_DIR}")

    # epiCRealism LoRAs
    if os.path.exists(EPIC_MODELS_DIR):
        print(f"Scanning epiCRealism: {EPIC_MODELS_DIR}")
        for root, _, files in os.walk(EPIC_MODELS_DIR):
            for f in files:
                if ".safe" in f.lower() or f.endswith(".bin"):
                    rel_path = os.path.relpath(os.path.join(root, f), EPIC_MODELS_DIR)
                    items.append((f"epiCRealism: {rel_path}", "epiCRealism", rel_path))
    else:
        print(f"✗ epiCRealism folder not found: {EPIC_MODELS_DIR}")

    # SD 3.5 LoRAs
    if os.path.exists(SD35_LORA_PATH):
        print(f"Scanning SD 3.5 : {SD35_LORA_PATH}")
        for root, _, files in os.walk(SD35_LORA_PATH):
            for f in files:
                if ".safe" in f.lower() or f.endswith(".bin"):
                    full_path = os.path.join(root, f)
                    rel_path = os.path.relpath(full_path, SD35_LORA_PATH)
                    items.append((f"SD 3.5: {rel_path}", "SD 3.5", full_path))
    else:
        print(f"✗ SD 3.5 folder not found: {SD35_LORA_PATH}")

    print(f"\n--- LoRA Summary ---")
    print(f"SD 1.5 LoRAs     : {sum(1 for i in items if i[1] == 'SD 1.5')}")
    print(f"epiCRealism LoRAs: {sum(1 for i in items if i[1] == 'epiCRealism')}")
    print(f"SD 3.5 LoRAs     : {sum(1 for i in items if i[1] == 'SD 3.5')}")
    print(f"Total            : {len(items)}")

    if not items:
        print("DEBUG: No models found at all.")
    return items


# Initial list check and default engine load
lora_list = get_lora_list()
load_engine("epiCRealism")  # Load epiCRealism as default since it has the most LoRAs

Scanning SD 1.5 : /kaggle/input/datasets/fatima753/respace-ai-models/models/sd1.5
Scanning epiCRealism: /kaggle/input/datasets/fatima753/respace-ai-models/models/epicrealism
Scanning SD 3.5 : /kaggle/input/datasets/fatima753/respace-ai-models/sd35-medium-turbo/sd35-medium-turbo

--- LoRA Summary ---
SD 1.5 LoRAs     : 3
epiCRealism LoRAs: 3
SD 3.5 LoRAs     : 3
Total            : 9
Switching engine to epiCRealism...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--emilianJR--epiCRealism/snapshots/6522cf856b8c8e14638a0aaa7bd89b1b098aed17/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances

✓ Engine loaded: epiCRealism


In [32]:
# 4. Define Inference Pipeline and UI Logic
def generate_interior(
    model_choice, room, use_case, theme, palette, lighting, furniture, 
    steps, guidance, seed
):
    # Parse choice: (Display Name, Engine Type, Lora Name)
    lora_list = get_lora_list()
    selected = [x for x in lora_list if x[0] == model_choice]
    if not selected: return None, "Model not found"
    
    _, engine_type, lora_name = selected[0]
    
    # Load model/lora
    status = load_lora_to_pipe(engine_type, lora_name)
    if "Error" in status: return None, status

    # Construct Prompt
    prompt_parts = [
        "intdesign" if engine_type == "SD 1.5" else "", # Trigger for SD1.5
        f"a photorealistic shot of a {room}",
        f"designed for a {use_case} atmosphere" if use_case else "",
        f"The aesthetic is defined by {theme}" if theme else "",
        f"and a {palette} color palette" if palette else "",
        f"under {lighting}" if lighting else "",
        f"featuring {', '.join(furniture)}" if furniture else "",
        "cinematic, 8k"
    ]
    prompt = ", ".join([p for p in prompt_parts if p])
    negative_prompt = "blurry, low quality, distorted, watermark, text, signature, lowres"

    print(f"Generating ({engine_type}) with: {prompt}")
    
    generator = torch.Generator(device).manual_seed(int(seed)) if seed > -1 else None
    
    try:
        # Step logic slightly different for Turbo-based SD3.5
        image = pipe(
            prompt, 
            negative_prompt=negative_prompt if engine_type == "SD 1.5" else None,
            num_inference_steps=int(steps), 
            guidance_scale=guidance,
            generator=generator
        ).images[0]
        return image, f"Status: {status}\nPrompt: {prompt}"
    except Exception as e:
        return None, f"Generation Error: {e}"

def update_use_case_list(room):
    return gr.Dropdown(choices=USE_CASES.get(room, []))


In [33]:
# 5 & 6. Create Gradio Web Interface and Launch
def update_model_dropdown():
    new_choices = [x[0] for x in get_lora_list()]
    return gr.Dropdown(choices=new_choices, value=new_choices[0] if new_choices else None)

with gr.Blocks(title="ReSpace-AI Multi-Engine Inference") as demo:
    gr.Markdown("# ReSpace-AI: Interior Design Generator")
    gr.Markdown("Switch between SD 1.5 and SD 3.5 Turbo models. Scans all subdirectories for models.")
    
    with gr.Row():
        with gr.Column(scale=1):
            # Model selection
            lora_choices = [x[0] for x in get_lora_list()]
            model_drop = gr.Dropdown(label="Select Model / Engine", choices=lora_choices, value=lora_choices[0] if lora_choices else None)
            refresh_btn = gr.Button("🔄 Refresh All Models")
            
            # Parameters
            room_drop = gr.Dropdown(label="Room Type", choices=ROOM_TYPES, value=ROOM_TYPES[0])
            use_case_drop = gr.Dropdown(label="Use Case", choices=USE_CASES[ROOM_TYPES[0]])
            theme_drop = gr.Dropdown(label="Color Theme", choices=COLOR_THEMES)
            palette_drop = gr.Dropdown(label="Color Palette", choices=COLOR_PALETTES)
            lighting_drop = gr.Dropdown(label="Lighting", choices=LIGHTING_CONDITIONS)
            furniture_check = gr.CheckboxGroup(label="Furniture", choices=FURNITURE_TYPES)
            
            with gr.Accordion("Advanced Settings", open=False):
                steps_slider = gr.Slider(minimum=1, maximum=50, step=1, value=25, label="Steps (Keep ~24 for SD3.5 Turbo)")
                cfg_slider = gr.Slider(minimum=1, maximum=15, step=0.5, value=7.0, label="Guidance Scale")
                seed_input = gr.Number(value=-1, label="Seed (-1 for random)")
            
            generate_btn = gr.Button("Generate Design", variant="primary")
            
        with gr.Column(scale=2):
            output_img = gr.Image(label="Generated Result")
            output_text = gr.Textbox(label="Status/Prompt", interactive=False)

    # Wire up events
    room_drop.change(fn=update_use_case_list, inputs=room_drop, outputs=use_case_drop)
    
    refresh_btn.click(
        fn=update_model_dropdown, 
        outputs=model_drop
    )
    
    generate_btn.click(
        fn=generate_interior,
        inputs=[
            model_drop, room_drop, use_case_drop, theme_drop, palette_drop, 
            lighting_drop, furniture_check, steps_slider, cfg_slider, seed_input
        ],
        outputs=[output_img, output_text]
    )

# Launch
demo.launch(share=True)

Scanning SD 1.5 : /kaggle/input/datasets/fatima753/respace-ai-models/models/sd1.5
Scanning epiCRealism: /kaggle/input/datasets/fatima753/respace-ai-models/models/epicrealism
Scanning SD 3.5 : /kaggle/input/datasets/fatima753/respace-ai-models/sd35-medium-turbo/sd35-medium-turbo

--- LoRA Summary ---
SD 1.5 LoRAs     : 3
epiCRealism LoRAs: 3
SD 3.5 LoRAs     : 3
Total            : 9
* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://2891ae25a34520b562.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Scanning SD 1.5 : /kaggle/input/datasets/fatima753/respace-ai-models/models/sd1.5
Scanning epiCRealism: /kaggle/input/datasets/fatima753/respace-ai-models/models/epicrealism
Scanning SD 3.5 : /kaggle/input/datasets/fatima753/respace-ai-models/sd35-medium-turbo/sd35-medium-turbo

--- LoRA Summary ---
SD 1.5 LoRAs     : 3
epiCRealism LoRAs: 3
SD 3.5 LoRAs     : 3
Total            : 9


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


Generating (epiCRealism) with: a photorealistic shot of a Bedroom, designed for a Kids atmosphere, The aesthetic is defined by Modern White, and a Neutral color palette, under Natural Daylight, featuring Lighting Fixtures, Plants, Sofas, cinematic, 8k


  0%|          | 0/25 [00:00<?, ?it/s]

Scanning SD 1.5 : /kaggle/input/datasets/fatima753/respace-ai-models/models/sd1.5
Scanning epiCRealism: /kaggle/input/datasets/fatima753/respace-ai-models/models/epicrealism
Scanning SD 3.5 : /kaggle/input/datasets/fatima753/respace-ai-models/sd35-medium-turbo/sd35-medium-turbo

--- LoRA Summary ---
SD 1.5 LoRAs     : 3
epiCRealism LoRAs: 3
SD 3.5 LoRAs     : 3
Total            : 9


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


Generating (epiCRealism) with: a photorealistic shot of a Bedroom, designed for a Kids atmosphere, The aesthetic is defined by Modern White, and a Neutral color palette, under Natural Daylight, featuring Lighting Fixtures, Plants, Sofas, Bed, cinematic, 8k


  0%|          | 0/25 [00:00<?, ?it/s]